# ELLA VIDEO ENGINE — Wan 2.2 TI2V 5B\n\n**Kaggle GPU notebook | Text-to-Video + Image-to-Video**\n\n**Protected fast-start edition:** the original notebook is backed up in `notebooks/backups/`. A verified `sd-cli` binary is reused without running CMake or compiling again. New builds detect the actual GPU architecture, preserve the build directory, and compile only `sd-cli`. Set **Accelerator = GPU** and **Internet = On**. Kaggle session and GPU quotas still apply.\n

In [ ]:
import os, subprocess, shutil, json, textwrap, pathlib, time, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert shutil.which('nvidia-smi'), 'Enable a Kaggle GPU accelerator in Notebook settings.'
required = {'git': 'git', 'cmake': 'cmake', 'g++': 'build-essential', 'ffmpeg': 'ffmpeg'}
missing = [pkg for cmd, pkg in required.items() if shutil.which(cmd) is None]
if missing:
    print('Installing only missing system tools:', missing, flush=True)
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git', 'cmake', 'build-essential', 'ffmpeg', 'libopenblas-dev'], check=True)
try:
    import huggingface_hub  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
print('ELLA environment ready; existing packages are reused.')


## 1. Build or reuse the CUDA runtime
This cell reuses the current build, compiles only `sd-cli` (not the unused server), uses a safe parallel job count, and repairs Kaggle's missing `CUDA::cuda_driver` CMake target before linking.

In [ ]:
%cd /kaggle/working
import torch

REPO_DIR = pathlib.Path('/kaggle/working/stable-diffusion.cpp')
BUILD_DIR = REPO_DIR / 'build'
SDCLI_PATH = BUILD_DIR / 'bin' / 'sd-cli'

# IMPORTANT: reuse a working binary immediately. Do not reconfigure or rebuild it.
binary_ready = SDCLI_PATH.is_file()
if binary_ready:
    try:
        check = subprocess.run([str(SDCLI_PATH), '--help'], capture_output=True, text=True, timeout=20)
        binary_ready = (check.returncode == 0)
    except Exception:
        binary_ready = False

if binary_ready:
    print('ELLA: existing sd-cli binary verified; skipping clone, CMake and compilation.', flush=True)
else:
    # Recover an interrupted/incomplete checkout safely.
    if not (REPO_DIR / '.git').exists() or not (REPO_DIR / 'CMakeLists.txt').is_file():
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        subprocess.run([
            'git', 'clone', '--depth', '1', '--recurse-submodules',
            'https://github.com/leejet/stable-diffusion.cpp.git', str(REPO_DIR)
        ], check=True)
    else:
        print('Using existing source checkout.', flush=True)

    CMAKE_FILE = REPO_DIR / 'CMakeLists.txt'
    cmake_text = CMAKE_FILE.read_text(encoding='utf-8')
    patch_marker = '# ELLA_KAGGLE_CUDA_DRIVER_TARGET'
    cuda_driver_patch = r'''
# ELLA_KAGGLE_CUDA_DRIVER_TARGET
if(NOT TARGET CUDA::cuda_driver)
  find_library(ELLA_CUDA_DRIVER_LIBRARY NAMES cuda
    HINTS
      "${CUDAToolkit_LIBRARY_DIR}"
      "${CUDAToolkit_ROOT}/lib64"
      "${CUDAToolkit_ROOT}/targets/x86_64-linux/lib/stubs"
      "/usr/local/cuda/lib64/stubs"
      "/usr/local/cuda/targets/x86_64-linux/lib/stubs"
      "/usr/local/cuda/compat"
      "/usr/lib/x86_64-linux-gnu"
  )
  if(NOT ELLA_CUDA_DRIVER_LIBRARY)
    find_file(ELLA_CUDA_DRIVER_LIBRARY NAMES libcuda.so.1
      PATHS "/usr/lib/x86_64-linux-gnu" "/usr/local/cuda/compat"
    )
  endif()
  if(NOT ELLA_CUDA_DRIVER_LIBRARY)
    message(FATAL_ERROR "ELLA: CUDA driver library (libcuda.so or libcuda.so.1) was not found.")
  endif()
  add_library(CUDA::cuda_driver UNKNOWN IMPORTED GLOBAL)
  set_target_properties(CUDA::cuda_driver PROPERTIES IMPORTED_LOCATION "${ELLA_CUDA_DRIVER_LIBRARY}")
endif()
'''
    if patch_marker not in cmake_text:
        anchor = 'find_package(CUDAToolkit REQUIRED)'
        if anchor not in cmake_text:
            raise RuntimeError('Could not locate CUDAToolkit section; refusing to patch an unknown CMake layout.')
        CMAKE_FILE.write_text(cmake_text.replace(anchor, anchor + '\n' + cuda_driver_patch, 1), encoding='utf-8')
        print('Applied the guarded Kaggle CUDA driver compatibility patch.', flush=True)
    else:
        print('CUDA driver compatibility patch already present.', flush=True)

    # Compile for the actual Kaggle GPU, not a hard-coded T4 (sm_75).
    major, minor = torch.cuda.get_device_capability(0)
    gpu_arch = f'{major}{minor}'
    print('Detected GPU:', torch.cuda.get_device_name(0), '| CUDA architecture:', gpu_arch, flush=True)

    configure = [
        'cmake', '-S', str(REPO_DIR), '-B', str(BUILD_DIR),
        '-DCMAKE_BUILD_TYPE=Release', '-DSD_CUDA=ON',
        '-DSD_CUDA_CUTLASS_PATH:PATH=',
        f'-DCMAKE_CUDA_ARCHITECTURES={gpu_arch}',
    ]
    subprocess.run(configure, check=True)

    mem_kb = 0
    try:
        for line in pathlib.Path('/proc/meminfo').read_text().splitlines():
            if line.startswith('MemAvailable:'):
                mem_kb = int(line.split()[1])
                break
    except Exception:
        pass
    jobs = min(os.cpu_count() or 2, 4)
    if mem_kb and mem_kb < 12 * 1024 * 1024:
        jobs = min(jobs, 2)

    print(f'Building only sd-cli with {jobs} jobs. Existing CMake objects are reused.', flush=True)
    subprocess.run([
        'cmake', '--build', str(BUILD_DIR), '--config', 'Release',
        '--target', 'sd-cli', '--parallel', str(jobs)
    ], check=True)

    if not SDCLI_PATH.is_file():
        raise RuntimeError(f'sd-cli build finished without producing {SDCLI_PATH}')
    subprocess.run([str(SDCLI_PATH), '--help'], check=True, stdout=subprocess.DEVNULL, timeout=20)

SDCLI = str(SDCLI_PATH)
print('ELLA CUDA runtime ready:', SDCLI, flush=True)


## 2. Download the model components (Q4)
The generator alone is not enough: Wan also requires its matching VAE and UMT5 text encoder.

In [ ]:
from huggingface_hub import hf_hub_download
MODEL_DIR = pathlib.Path('/kaggle/working/ella_models'); MODEL_DIR.mkdir(exist_ok=True)
def get_file(repo, name):
    target = MODEL_DIR / name
    if not target.exists():
        hf_hub_download(repo_id=repo, filename=name, local_dir=str(MODEL_DIR))
    assert target.exists(), f'Missing download: {target}'
    print(name, round(target.stat().st_size/1e9, 2), 'GB')
get_file('off-grid-ai/Wan2.2-TI2V-5B-GGUF', 'Wan2.2-TI2V-5B-Q4_K_S.gguf')
get_file('off-grid-ai/Wan2.2-TI2V-5B-GGUF', 'wan2.2_vae.safetensors')
get_file('off-grid-ai/Video-Companions-GGUF', 'umt5-xxl-encoder-Q4_K_M.gguf')


## 3. Fast validation profile
Use a smaller first-test profile on Kaggle T4 to validate the full pipeline before increasing quality. The worker uses the same defaults.

In [ ]:
PROMPT = 'A cinematic realistic scene: a young woman walks slowly through a rainy Istanbul street at night, reflections on the pavement, natural human movement, dramatic film lighting, smooth camera tracking.'
NEGATIVE_PROMPT = 'low quality, blurry, distorted anatomy, extra fingers, flicker, subtitles, watermark, static frame'
MODE = 'text'  # use 'image' for image-to-video
REFERENCE_IMAGE = '/kaggle/working/reference.png'
DURATION_SECONDS = 5
FPS = 8
WIDTH, HEIGHT = 832, 480
STEPS = 8
SEED = 42
print('Fast validation settings ready:', WIDTH, 'x', HEIGHT, '|', STEPS, 'steps |', DURATION_SECONDS, 'seconds')


In [ ]:
!mkdir -p /kaggle/working/ella_engine
print('ELLA runner will be fetched from the project branch in the next cell.')


## 4. Generate MP4
This cell calls the ELLA runner with argument-safe subprocess execution (no shell interpolation).

In [ ]:
import importlib.util
RUNNER_PATH = pathlib.Path('/kaggle/working/ella_engine/runner.py')
if not RUNNER_PATH.exists():
    import urllib.request
    url='https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/ella-wan-engine/runner.py'
    urllib.request.urlretrieve(url, RUNNER_PATH)
spec=importlib.util.spec_from_file_location('ella_runner', RUNNER_PATH)
runner=importlib.util.module_from_spec(spec); spec.loader.exec_module(runner)
args=runner.parser().parse_args(['--prompt', PROMPT, '--negative-prompt', NEGATIVE_PROMPT, '--duration', str(DURATION_SECONDS), '--fps', str(FPS), '--width', str(WIDTH), '--height', str(HEIGHT), '--steps', str(STEPS), '--seed', str(SEED), '--output', '/kaggle/working/ella_output.mp4'])
args.sd_cli=pathlib.Path(SDCLI)
args.diffusion_model=MODEL_DIR/'Wan2.2-TI2V-5B-Q4_K_S.gguf'
args.vae=MODEL_DIR/'wan2.2_vae.safetensors'
args.text_encoder=MODEL_DIR/'umt5-xxl-encoder-Q4_K_M.gguf'
if MODE == 'image':
    args.image=pathlib.Path(REFERENCE_IMAGE)
result=runner.run(args)
print(json.dumps(result, ensure_ascii=False, indent=2))
from IPython.display import Video, display
display(Video('/kaggle/working/ella_output.mp4', embed=True))


## 5. Download your MP4
The final file is `/kaggle/working/ella_output.mp4`. Use the Kaggle file browser to download it. Keep the generation log (`ella_output.log`) if anything fails.

In [ ]:
!ls -lh /kaggle/working/ella_output.mp4 /kaggle/working/ella_output.log


## 6. Optional: connect the worker to Ella2027
Run this cell **only after the manual MP4 test succeeds**. It needs the Kaggle Secret `ELLA_KAGGLE_WAN22_WORKER_TOKEN`, whose value must match Render's `KAGGLE_WAN22_WORKER_TOKEN`. If you already have the older `ELLA_KAGGLE_WORKER_TOKEN`, the cell can fall back to it. The worker then polls Ella and processes queued scenes one by one. Do not paste the token into notebook code.

In [ ]:
import urllib.request, sys
WORKER_PATH = pathlib.Path('/kaggle/working/ella_engine/wan22_worker.py')
WORKER_URL = 'https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/kaggle-wan22-worker/wan22_worker.py'
urllib.request.urlretrieve(WORKER_URL, WORKER_PATH)
sys.path.insert(0, str(WORKER_PATH.parent))
from kaggle_secrets import UserSecretsClient
try:
    ELLA_TOKEN = UserSecretsClient().get_secret('ELLA_KAGGLE_WAN22_WORKER_TOKEN')
except Exception:
    ELLA_TOKEN = UserSecretsClient().get_secret('ELLA_KAGGLE_WORKER_TOKEN')
from wan22_worker import run_forever
run_forever('https://ella2027.onrender.com', ELLA_TOKEN, RUNNER_PATH, MODEL_DIR, pathlib.Path(SDCLI), pathlib.Path('/kaggle/working/ella_jobs'), max_scene_seconds=5)
